# <u>K-means and K-medoids</u>

### Prerequisites:
* <a href="../4.Clustering/Hierarchical Clustering.ipynb">Check out the notebook on Hierarchical Clustering</a>

## Topics
* [1. K-means](#kmeans)
    * [1.1 Hierarchical Clustering vs Partition clustering](#partition)
    * [1.2 Objective Function](#objective)
    * [1.3 Algorithm (Iterative Approximation)](#alg)
    * [1.4 Key Properties](#key)
    * [1.5 Choosing K](#K)
    * [1.6 Intuition](#intuition)
    * [1.7 K-means library](#library)

* [2. K-medoids](#kmedoids)
    * [2.1 Key Idea](#idea)
    * [2.2 Algorithm (PAM – Partitioning Around Medoids)](#pam)
    * [2.3 Distance Metrics](#metric)
    * [2.4 Properties](#properties)
    * [2.5 Comparison to K-means](#compare)
    * [2.6 Choosing K](#k)
    * [2.7 Intuition](#intuition2)
    * [2.8 K-medoids library](#lib)

* [3. Cluster Validation Measure for k-means and k-medoids](#eval)
* [4. Summary](#summary)

In [4]:
import numpy as np # for math and random numbers
import plotly.express as px # for plotting
import plotly.graph_objects as go # for plotting
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs # make data for classification
from sklearn.preprocessing import StandardScaler # for standardizing features
from sklearn.cluster import KMeans # for k-means
#from sklearn_extra.cluster import KMedoids
print("Setup complete")

Setup complete


<a class="anchor" id="kmeans"></a>
# 1. K-means

<a class="anchor" id="partition"></a>
## 1.1 Hierarchical Clustering vs Partition clustering

<div style="display:flex; gap:20px;">
<!--  -->
<div style="
padding:16px;
border-radius:8px;
width:50%;
">

#### Hierarchical clustering
- Stepwise merging (agglomerative HC) or dividing (divisive HC) of clusters based on distances and linkages
- Number of clusters are selected by splitting the dendrogram at a specific height after visual inspection.

</div>


<!--  -->
<div style="
padding:16px;
border-radius:8px;
width:50%;
">

#### Partitioning clustering
- Partitions observations into a predefined number of K clusters by optimizing a numerical criterion
-  most common method is K-means, which uses centroids, i.e., artificial data points located at the mean value of each dimension (works only for numerical data)

</div>
</div>

<a class="anchor" id="objective"></a>
## 1.2 Objective Function

**Goal: Partition N observations into K predefined clusters $C_1,\ldots,C_K$ such that points within a cluster (within-cluster variation) are as close as possible (minimize compactness).**

K-means solves: $$\min_K \sum_{k=1}^K \sum_{i \in C_k} \lVert X_i - \bar{X}_k \rVert_2^2$$

- centroid of cluster $k$: $\bar{X}_k = \frac{1}{N_k} \sum_{i \in C_k}X_i$
- number of observations in cluster $k$: $N_k$

<p align="center">
<img src="pics\17.png" width="600"/>
</p>

<a class="anchor" id="alg"></a>
## 1.3 Algorithm (Iterative Approximation)

**Idea:** Consider all possible partitions of N observations into K clusters and select the cluster witht lowest within-cluster variation

**Problem:** Requires trying all possible assignments of N observations into K clusters,which in practice is nearly impossible:

$$
\begin{array}{ccc}
N & K & \text{number of possible partitions} \\
\hline
15 & 3 &  2.375.101 \\
20 & 4 & 45.232.115.901 \\
100 & 5 & 10^{68} \\
\end{array}
$$


**Solutuion:** Since finding the exact optimum is computationally infeasible, K-means uses an iterative procedure:

1. **Initialization:** <br>
    Choose \(K\) random points as initial cluster centers

2. **Assignment step:** <br>
    Assign each point to the **nearest centroid**

3. **Update step:** <br>
    Recompute each centroid as the **mean of its assigned points**

4. **Repeat** <br>
    Continue until centroids no longer change

<a class="anchor" id="key"></a>
## 1.4 Key Properties

- &#9989; Always converges (objective decreases each iteration)
- &#128680; Depends on initialization $\rightarrow$ may find only a local optimum
- &#128257; Often run multiple times (different starts) and pick best result
- &#128202; Works only for numerical data (uses means)
- &#128680; Sensitive to outliers (means get distorted) 

<a class="anchor" id="K"></a>
## 1.5 Choosing K

- Increasing K always reduces within-cluster variation
- Use elbow method: Look for a "bend" in the curve of variation vs. K

<p align="center">
<img src="pics\18.png" width="600"/>
</p>

<a class="anchor" id="intuition"></a>
## 1.6 Intuition

- Clusters are represented by centroids (means)
- Each iteration:
    - points move to closest center
    - centers move to the “middle” of their points

$\Rightarrow$ Eventually stabilizes into a partition of the data

<a class="anchor" id="library"></a>
## 1.7 K-means library

```python
# 1. Import K-Means
from sklearn.cluster import KMeans

# 2. Basic K-Means Model
kmeans = KMeans(
    n_clusters=3, # number of clusters K
    init="k-means++", # initialization method ("random" or "k-means++")
    n_init=10, # number of restarts
    max_iter=300, # max iterations
    random_state=1125
)

labels = kmeans.fit_predict(X) # cluster assignments
centers = kmeans.cluster_centers_ # centroids


# 3. Visualize Clusters (2D)
import matplotlib.pyplot as plt

plt.scatter(X[:, 0], X[:, 1], c=labels)
plt.scatter(centers[:, 0], centers[:, 1], marker="x")  # centroids
plt.title("K-Means Clustering")
plt.show()


# 4. Elbow Method (choose K)
inertia = [] # within-cluster sum of squares

K_range = range(1, 10)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X)
    inertia.append(km.inertia_)

plt.plot(K_range, inertia, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Within-cluster variation (inertia)")
plt.title("Elbow Method")
plt.show()


# 5. Silhouette Score (cluster quality)
from sklearn.metrics import silhouette_score

for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels_k = km.fit_predict(X)
    score = silhouette_score(X, labels_k)
    print(f"K={k}, Silhouette Score={score:.3f}")


# 6. Standardization (important!)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans_scaled = KMeans(n_clusters=3, random_state=42)
labels_scaled = kmeans_scaled.fit_predict(X_scaled)


# 7. Pipeline (with preprocessing)
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(n_clusters=3, random_state=42))
])

labels_pipeline = pipeline.fit_predict(X)


# 8. Multiple Initializations (robustness)
kmeans_multi = KMeans(
    n_clusters=3,
    n_init=50, # more restarts → better chance of global optimum
    random_state=1127
)

labels_multi = kmeans_multi.fit_predict(X)


# 9. Predict Cluster for New Data
new_points = [[0, 0], [5, 5]]
predictions = kmeans.predict(new_points)


# 10. Compare Different K Values
for k in [2, 3, 4, 5]:
    km = KMeans(n_clusters=k, random_state=42)
    labels_k = km.fit_predict(X)
    print(f"K={k}, Inertia={km.inertia_:.2f}")


# 11. Toy Example
from sklearn.datasets import make_blobs

X, y = make_blobs(
    n_samples=100,
    centers=3,
    n_features=2,
    random_state=1415
)

kmeans = KMeans(n_clusters=3, random_state=42)
labels = kmeans.fit_predict(X)

plt.scatter(X[:, 0], X[:, 1], c=labels)
plt.scatter(kmeans.cluster_centers_[:, 0],
            kmeans.cluster_centers_[:, 1],
            marker="x")
plt.title("Toy Data K-Means")
plt.show()


# 12. Key Attributes
print("Cluster centers:\n", kmeans.cluster_centers_)
print("Labels:\n", kmeans.labels_[:10])
print("Inertia:", kmeans.inertia_) # within-cluster variation


# 13. Notes
# - Works only with numerical data
# - Sensitive to outliers
# - Depends on initialization -> use multiple runs (n_init)
# - Always converges, but may reach local optimum
# - Choose K via elbow or silhouette method
```

<a class="anchor" id="kmedoids"></a>
# 2. K-medoids

<a class="anchor" id="idea"></a>
## 2.1 Key Idea

**Like K-means, K-medoids partitions data into K clusters, but uses real data points (medoids) as cluster centers instead of artificial means.**

- A medoid is the data point in a cluster with the smallest average distance to all other points in that cluster
- Clusters are represented by actual observations, not computed averages

<a class="anchor" id="pam"></a>
## 2.2 Algorithm (PAM – Partitioning Around Medoids)

1. **Initialization:** <br>
Randomly select K data points as medoids

2. **Assignment step:** <br>
Assign each point to the closest medoid

3. **Update step:** <br>
    - Swap a medoid with another point
    - Compute total within-cluster distance
    - Keep the configuration with the lowest variation

4. **Repeat until medoids do not change**

<a class="anchor" id="metric"></a>
## 2.3 Distance Metrics

Common choices:
- Euclidean distance
- Manhattan distance (more robust to outliers)

<a class="anchor" id="properties"></a>
## 2.4 Properties
- &#9989; Robust to outliers (unlike K-means)
- &#9989; Works with categorical and non-numeric data
- &#128680; More computationally expensive than K-means
- &#128680; Depends on initialization $\rightarrow$ may reach local optimum

<a class="anchor" id="compare"></a>
## 2.5 Comparison to K-means

| Feature        | K-means           | K-medoids                 |
| -------------- | ----------------- | ------------------------- |
| Cluster center | Mean (artificial) | Real data point (medoid)  |
| Outliers       | Sensitive         | Robust                    |
| Data type      | Numeric only      | Also categorical possible |
| Speed          | Faster            | Slower                    |


<a class="anchor" id="k"></a>
## 2.6 Choosing K

- Must be specified beforehand
- Can use methods like the silhouette score to evaluate clustering quality

<a class="anchor" id="intuition2"></a>
## 2.7 Intuition

- Instead of moving a "center" (mean), K-medoids:
    - chooses representative points
    - and improves clusters by swapping candidates

$\Rightarrow$ This makes it more stable when data contains noise or extreme values

<a class="anchor" id="lib"></a>
## 2.8 K-medoids library

```python
# 1. Import K-Medoids (scikit-learn-extra)
# pip install scikit-learn-extra (if not installed)
from sklearn_extra.cluster import KMedoids

# 2. Basic K-Medoids Model
kmedoids = KMedoids(
    n_clusters=3, # number of clusters K
    metric="euclidean", # distance metric ("euclidean", "manhattan", etc.)
    init="k-medoids++", # initialization ("random", "heuristic", "k-medoids++")
    max_iter=300,
    random_state=1125
)

labels = kmedoids.fit_predict(X) # cluster assignments
medoids = kmedoids.cluster_centers_ # medoids (actual data points)


# 3. Visualize Clusters (2D)
import matplotlib.pyplot as plt

plt.scatter(X[:, 0], X[:, 1], c=labels)
plt.scatter(medoids[:, 0], medoids[:, 1], marker="x")  # medoids
plt.title("K-Medoids Clustering")
plt.show()


# 4. Evaluate Different K (using inertia-like cost)
costs = []

K_range = range(1, 10)
for k in K_range:
    km = KMedoids(n_clusters=k, random_state=42)
    km.fit(X)
    costs.append(km.inertia_)  # sum of distances to medoids

plt.plot(K_range, costs, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Total distance (cost)")
plt.title("Elbow-like Method for K-Medoids")
plt.show()


# 5. Silhouette Score (cluster quality)
from sklearn.metrics import silhouette_score

for k in range(2, 7):
    km = KMedoids(n_clusters=k, random_state=42)
    labels_k = km.fit_predict(X)
    score = silhouette_score(X, labels_k)
    print(f"K={k}, Silhouette Score={score:.3f}")


# 6. Standardization (important!)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmedoids_scaled = KMedoids(n_clusters=3, random_state=42)
labels_scaled = kmedoids_scaled.fit_predict(X_scaled)


# 7. Pipeline (with preprocessing)
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("kmedoids", KMedoids(n_clusters=3, random_state=42))
])

labels_pipeline = pipeline.fit_predict(X)


# 8. Compare Distance Metrics
for metric in ["euclidean", "manhattan", "cosine"]:
    km = KMedoids(n_clusters=3, metric=metric, random_state=42)
    labels_m = km.fit_predict(X)
    print(f"Metric={metric}, Cost={km.inertia_:.2f}")


# 9. Predict Cluster for New Data
new_points = [[0, 0], [5, 5]]
predictions = kmedoids.predict(new_points)


# 10. Compare Different K Values
for k in [2, 3, 4, 5]:
    km = KMedoids(n_clusters=k, random_state=42)
    labels_k = km.fit_predict(X)
    print(f"K={k}, Cost={km.inertia_:.2f}")


# 11. Toy Example
from sklearn.datasets import make_blobs

X, y = make_blobs(
    n_samples=100,
    centers=3,
    n_features=2,
    random_state=1415
)

kmedoids = KMedoids(n_clusters=3, random_state=42)
labels = kmedoids.fit_predict(X)

plt.scatter(X[:, 0], X[:, 1], c=labels)
plt.scatter(kmedoids.cluster_centers_[:, 0],
            kmedoids.cluster_centers_[:, 1],
            marker="x")
plt.title("Toy Data K-Medoids")
plt.show()


# 12. Key Attributes
print("Medoids:\n", kmedoids.cluster_centers_)
print("Labels:\n", kmedoids.labels_[:10])
print("Cost (sum of distances):", kmedoids.inertia_)


# 13. Notes
# - Uses real data points as cluster centers (medoids)
# - More robust to outliers than K-Means
# - Works with arbitrary distance metrics (incl. categorical data)
# - Slower than K-Means (more computationally expensive)
# - Depends on initialization -> may reach local optimum
# - Choose K via silhouette or elbow-like method
```

<a class="anchor" id="eval"></a>
# 7. Cluster Validation Measure for k-means and k-medoids

Clustering goal: 
- (Average) distance within all clusters should be small
- (Average) distance between clusters should be large

<p align="center">
<img src="pics\0.jpeg" width="600"/>
</p>
Requirements for Validtation measures
- compactness measure to evaluate how close obseervations within clusters are
- separation measure to determine how well separated clusters are from other clusters

- combine both measures with weights $\alpha$ and $\beta$ as $$\text{index} = \frac{\alpha \cdot \text{separation}}{\beta \cdot \text{compactness}}$$ 

### Dunn index

$$
D=\frac{\text{separation}}{\text{compactness}}=\frac{\min_{i\neq j}d(C_i,C_j)}{\max_{k}d(\Delta C_k)}
$$

<p align="center">
<img src="pics\16.png" width="600"/>
</p>

- Separation $\min_{i\neq j}d(C_i,C_j)$
    - For each cluster compute the pairiwse distance between each observation inside a cluster and the observations in all other clusters
    - Use the minimum of those pairwise distances as separation measure

- Compactness $\max_{k}d(\Delta C_k)$
    - For each cluster compute the distance between the observation inside the same cluster 
    - Use the maximum of those distances as compactness measure


- Large $D$ indicates better clustering



### Silhouette coefficient

$$
S_i = \frac{b_i - a_i}{\max(a_i,b_i)}, \quad S_i \in [-1,1]
$$

- $a_i$ is the average distance between $i$ and all other observations of the cluster to which $i$ belongs
- $b_i=min_{C(i)\neq k}d(C(i),C_k)$
    - $d(C(i),C_k)$ is average distance between observation $i$ from cluster $C(i)$ to all observations in cluster $C_k$ to which $i$ does not belong
    - $b_i$ can be seen as the distance between $i$ and its closest cluster to which $i$ does not belong
- A high average silhouette value indicates better clustering


<a class="anchor" id="summary"></a>
# 3. Summary

## 1. K-means clustering
- Finding the exact minimum within-cluster variation is computationally difficult, so the K-means algorithm provides an approximate solution.
- K-means is guaranteed to converge, but:
    - the final clusters depend heavily on the initially chosen centers,
    - so it is common practice to run the algorithm multiple times with different initializations.
- The elbow method is a simple technique used to estimate the best number of clusters K.
## 2. Alternative partitioning methods
### K-medians
- Similar to K-means, but each cluster center is defined using the median instead of the mean.
- Better suited for ordinal data, but not appropriate for purely nominal/categorical data.
### K-medoids
- Uses actual observed data points called medoids as cluster centers rather than artificial average points.
- A medoid is the point with the smallest average distance to all other points in the cluster.
- Can also handle nominal/categorical data better than K-means.
- However, finding medoids is computationally more difficult.
## 3. Properties of K-medoids
- Considered a more robust alternative to K-means.
- Less sensitive to outliers/noise because centers are real observations.
- Each cluster center is an existing data point selected from the dataset.
- Like K-means, the user must choose the number of clusters K, or use methods such as the silhouette method to determine it.
- Both results and runtime depend strongly on the initial assignment of clusters.
- K-medoids, like K-means, may converge only to a local optimum, though better initialization strategies can improve performance.


### Overall takeaway
- K-means is fast and widely used, but sensitive to initialization and outliers.
- K-medians replaces means with medians for more robust central values.
- K-medoids is the most robust among the three because it uses real data points and handles outliers/categorical data better, though at a higher computational cost.